# 2.5D vs 3D VoxelMorph comparison

This notebook rebuilds the pretrained 2.5D (full and split) and 3D dense VoxelMorph models, loads their checkpoints from `../trained_weights`, and compares model size (parameter count + file size) alongside registration quality metrics (MSE, NCC, flow smoothness, and Dice).

All evaluation uses the two provided subjects in `../Data` and automatically leverages CUDA when available.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from IPython.display import display
from pathlib import Path

DATA_DIR = Path('../Data')
WEIGHT_DIR = Path('../trained_weights')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [2]:
subj1 = np.load(DATA_DIR / 'subj1.npz')
subj2 = np.load(DATA_DIR / 'subj2.npz')

volumes = np.stack([subj1['vol'], subj2['vol']]).astype('float32')
segmentations = np.stack([subj1['seg'], subj2['seg']]).astype('float32')

label_ids = sorted(set(np.unique(segmentations[0]).tolist() + np.unique(segmentations[1]).tolist()))
vol_shape = volumes.shape[1:]

volumes_tensor = torch.from_numpy(volumes).unsqueeze(1).to(device)
segmentations_tensor = torch.from_numpy(segmentations).unsqueeze(1).to(device)

print('Volumes tensor:', volumes_tensor.shape)
print('Segmentations tensor:', segmentations_tensor.shape)
print('Labels:', len(label_ids))

Volumes tensor: torch.Size([2, 1, 160, 192, 224])
Segmentations tensor: torch.Size([2, 1, 160, 192, 224])
Labels: 44


In [3]:
class SpatialTransformer2D(nn.Module):
    def __init__(self, mode='bilinear', padding_mode='border'):
        super().__init__()
        self.mode = mode
        self.padding_mode = padding_mode

    def forward(self, src, flow):
        b, _, h, w = src.shape
        device = src.device
        yy, xx = torch.meshgrid(
            torch.arange(h, device=device),
            torch.arange(w, device=device),
            indexing='ij'
        )
        grid = torch.stack((xx, yy), dim=0).float()
        grid = grid.unsqueeze(0).repeat(b, 1, 1, 1)
        pts = grid + flow
        x = 2.0 * (pts[:, 0] / (w - 1.0)) - 1.0
        y = 2.0 * (pts[:, 1] / (h - 1.0)) - 1.0
        sample_grid = torch.stack((x, y), dim=-1)
        return F.grid_sample(
            src,
            sample_grid,
            mode=self.mode,
            padding_mode=self.padding_mode,
            align_corners=True,
        )


class SpatialTransformer3D(nn.Module):
    def __init__(self, mode='bilinear', padding_mode='border'):
        super().__init__()
        self.mode = mode
        self.padding_mode = padding_mode

    def forward(self, src, flow):
        b, _, d, h, w = src.shape
        device = src.device
        zz, yy, xx = torch.meshgrid(
            torch.arange(d, device=device),
            torch.arange(h, device=device),
            torch.arange(w, device=device),
            indexing='ij'
        )
        grid = torch.stack((xx, yy, zz), dim=0).float()
        grid = grid.unsqueeze(0).repeat(b, 1, 1, 1, 1)
        pts = grid + flow
        x = 2.0 * (pts[:, 0] / (w - 1.0)) - 1.0
        y = 2.0 * (pts[:, 1] / (h - 1.0)) - 1.0
        z = 2.0 * (pts[:, 2] / (d - 1.0)) - 1.0
        sample_grid = torch.stack((x, y, z), dim=-1)
        return F.grid_sample(
            src,
            sample_grid,
            mode=self.mode,
            padding_mode=self.padding_mode,
            align_corners=True,
        )


class ConvBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x):
        return self.act(self.conv(x))


class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x):
        return self.act(self.conv(x))


class SliceRegistrationNet(nn.Module):
    def __init__(self, enc_feats, final_feats):
        super().__init__()
        self.encoders = nn.ModuleList()
        in_channels = 2
        for nf in enc_feats:
            self.encoders.append(ConvBlock2D(in_channels, nf))
            in_channels = nf
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock2D(enc_feats[-1], enc_feats[-1])
        self.decoders = nn.ModuleList()
        decoder_in = enc_feats[-1]
        for skip_ch in reversed(enc_feats):
            self.decoders.append(ConvBlock2D(decoder_in + skip_ch, skip_ch))
            decoder_in = skip_ch
        self.final_conv0 = ConvBlock2D(decoder_in, final_feats[0])
        self.final_conv1 = ConvBlock2D(final_feats[0], final_feats[1])
        self.flow = nn.Conv2d(final_feats[1], 2, kernel_size=3, padding=1)
        self.transformer = SpatialTransformer2D()
        self.apply(self._init_weights)
        nn.init.zeros_(self.flow.weight)
        nn.init.zeros_(self.flow.bias)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, nonlinearity='leaky_relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, src, tgt):
        x = torch.cat([src, tgt], dim=1)
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for skip, dec in zip(reversed(skips), self.decoders):
            x = F.interpolate(x, scale_factor=2, mode='nearest')
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        x = self.final_conv0(x)
        x = self.final_conv1(x)
        flow = self.flow(x)
        moved = self.transformer(src, flow)
        return moved, flow


class FlowFusionNet(nn.Module):
    def __init__(self, in_channels=6, hidden_channels=32):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv3d(in_channels, hidden_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv3d(hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
        )
        self.block2 = nn.Sequential(
            nn.Conv3d(hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
        )
        self.out = nn.Conv3d(hidden_channels, 3, kernel_size=1)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.out(x)


class VxmDense3D(nn.Module):
    def __init__(self, vol_shape, enc_feats=(16, 32, 32, 32), final_feats=(32, 16)):
        super().__init__()
        self.vol_shape = vol_shape
        self.encoders = nn.ModuleList()
        in_channels = 2
        for nf in enc_feats:
            self.encoders.append(ConvBlock3D(in_channels, nf))
            in_channels = nf
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = ConvBlock3D(enc_feats[-1], enc_feats[-1])
        self.decoders = nn.ModuleList()
        decoder_in = enc_feats[-1]
        for skip_ch in reversed(enc_feats):
            self.decoders.append(ConvBlock3D(decoder_in + skip_ch, skip_ch))
            decoder_in = skip_ch
        self.final_conv0 = ConvBlock3D(decoder_in, final_feats[0])
        self.final_conv1 = ConvBlock3D(final_feats[0], final_feats[1])
        self.flow = nn.Conv3d(final_feats[1], 3, kernel_size=3, padding=1)
        self.transformer = SpatialTransformer3D()
        self.apply(self._init_weights)
        nn.init.zeros_(self.flow.weight)
        nn.init.zeros_(self.flow.bias)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Conv3d):
            nn.init.kaiming_normal_(module.weight, nonlinearity='leaky_relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, src, tgt):
        x = torch.cat([src, tgt], dim=1)
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for skip, dec in zip(reversed(skips), self.decoders):
            x = F.interpolate(x, scale_factor=2, mode='trilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        x = self.final_conv0(x)
        x = self.final_conv1(x)
        flow = self.flow(x)
        moved = self.transformer(src, flow)
        return moved, flow

In [4]:
ORIENTATIONS = ('axial', 'coronal', 'sagittal')
SLICE_EVAL_BATCH = 16

def volume_to_slices(volume, orientation):
    vol = volume[0, 0]
    if orientation == 'axial':
        data = vol
    elif orientation == 'coronal':
        data = vol.permute(1, 0, 2)
    elif orientation == 'sagittal':
        data = vol.permute(2, 0, 1)
    else:
        raise ValueError(f'Unknown orientation: {orientation}')
    return data.unsqueeze(1)

def run_slice_orientation(slice_model, moving, fixed, orientation, batch_size=SLICE_EVAL_BATCH):
    flows = []
    moving_slices = volume_to_slices(moving, orientation)
    fixed_slices = volume_to_slices(fixed, orientation)
    total = moving_slices.shape[0]
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        moving_chunk = moving_slices[start:end].to(device)
        fixed_chunk = fixed_slices[start:end].to(device)
        with torch.no_grad():
            _, flow_chunk = slice_model(moving_chunk, fixed_chunk)
        flows.append(flow_chunk.detach())
    flow_slices = torch.cat(flows, dim=0)
    return lift_slice_flows(flow_slices, orientation)

def lift_slice_flows(flow_slices, orientation):
    if orientation == 'axial':
        flow_x = flow_slices[:, 0][None, None, ...]
        flow_y = flow_slices[:, 1][None, None, ...]
        flow_z = torch.zeros_like(flow_x)
    elif orientation == 'coronal':
        flow_x = flow_slices[:, 0].permute(1, 0, 2)[None, None, ...]
        flow_z = flow_slices[:, 1].permute(1, 0, 2)[None, None, ...]
        flow_y = torch.zeros_like(flow_x)
    else:
        flow_y = flow_slices[:, 0].permute(1, 2, 0)[None, None, ...]
        flow_z = flow_slices[:, 1].permute(1, 2, 0)[None, None, ...]
        flow_x = torch.zeros_like(flow_y)
    flow_volume = torch.cat([flow_x, flow_y, flow_z], dim=1)
    counts = torch.zeros_like(flow_volume)
    if orientation == 'axial':
        counts[:, 0] += 1.0
        counts[:, 1] += 1.0
    elif orientation == 'coronal':
        counts[:, 0] += 1.0
        counts[:, 2] += 1.0
    else:
        counts[:, 1] += 1.0
        counts[:, 2] += 1.0
    return flow_volume, counts

def fuse_flow_components(flows, counts):
    stacked = torch.stack(flows)
    stacked_counts = torch.stack(counts)
    return stacked.sum(dim=0) / stacked_counts.sum(dim=0).clamp(min=1.0)

def flow_grad_l2_3d(flow):
    dx = flow[:, :, :, :, 1:] - flow[:, :, :, :, :-1]
    dy = flow[:, :, :, 1:, :] - flow[:, :, :, :-1, :]
    dz = flow[:, :, 1:, :, :] - flow[:, :, :-1, :, :]
    dx = F.pad(dx, (0, 1, 0, 0, 0, 0))
    dy = F.pad(dy, (0, 0, 0, 1, 0, 0))
    dz = F.pad(dz, (0, 0, 0, 0, 0, 1))
    return dx.pow(2).mean() + dy.pow(2).mean() + dz.pow(2).mean()

def ncc(a, b, eps=1e-5):
    a_mean = a.mean()
    b_mean = b.mean()
    numer = ((a - a_mean) * (b - b_mean)).mean()
    denom = torch.sqrt(((a - a_mean) ** 2).mean() * ((b - b_mean) ** 2).mean() + eps)
    return (numer / denom).item()

def dice_from_flow(flow, moving_seg, fixed_seg, labels, warper):
    warped = warper(moving_seg.float(), flow).round()
    stats = {}
    for label in labels:
        moving_mask = (warped == label).float()
        fixed_mask = (fixed_seg == label).float()
        denom = moving_mask.sum() + fixed_mask.sum()
        if denom.item() == 0:
            continue
        stats[label] = (2.0 * (moving_mask * fixed_mask).sum() / denom).item()
    return stats


In [5]:
slice_full = SliceRegistrationNet(enc_feats=(12, 24, 24), final_feats=(32, 24)).to(device)
slice_split = SliceRegistrationNet(enc_feats=(32, 64, 64), final_feats=(64, 32)).to(device)
fusion_split = FlowFusionNet(in_channels=6, hidden_channels=32).to(device)
dense_3d = VxmDense3D(vol_shape=vol_shape, enc_feats=(16, 32, 32, 32), final_feats=(32, 16)).to(device)

full_slice_state = torch.load(WEIGHT_DIR / '2p5_full_slice.pth', map_location=device)
slice_full.load_state_dict(full_slice_state)

split_slice_state = torch.load(WEIGHT_DIR / '2p5_split_slice.pth', map_location=device)
slice_split.load_state_dict(split_slice_state)

split_fusion_state = torch.load(WEIGHT_DIR / '2p5_split_fusion.pth', map_location=device)
fusion_split.load_state_dict(split_fusion_state)

vxm3d_state = torch.load(WEIGHT_DIR / '3d_dense.pth', map_location=device)
dense_3d.load_state_dict(vxm3d_state)

for model in (slice_full, slice_split, fusion_split, dense_3d):
    model.eval()

intensity_warper = SpatialTransformer3D().to(device)
label_warper = SpatialTransformer3D(mode='nearest').to(device)
label_warper.eval()

SpatialTransformer3D()

In [6]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def weight_size_mb(path):
    return Path(path).stat().st_size / (1024 ** 2)

full_slice_path = WEIGHT_DIR / '2p5_full_slice.pth'
split_slice_path = WEIGHT_DIR / '2p5_split_slice.pth'
split_fusion_path = WEIGHT_DIR / '2p5_split_fusion.pth'
dense_path = WEIGHT_DIR / '3d_dense.pth'

size_rows = [
    {'model': '2.5D full', 'component': 'slice only', 'params_m': count_parameters(slice_full) / 1e6, 'weights_mb': weight_size_mb(full_slice_path)},
    {'model': '2.5D split', 'component': 'slice only', 'params_m': count_parameters(slice_split) / 1e6, 'weights_mb': weight_size_mb(split_slice_path)},
    {'model': '2.5D split', 'component': 'fusion only', 'params_m': count_parameters(fusion_split) / 1e6, 'weights_mb': weight_size_mb(split_fusion_path)},
    {'model': '2.5D split', 'component': 'slice + fusion', 'params_m': (count_parameters(slice_split) + count_parameters(fusion_split)) / 1e6, 'weights_mb': weight_size_mb(split_slice_path) + weight_size_mb(split_fusion_path)},
    {'model': '3D dense', 'component': 'full network', 'params_m': count_parameters(dense_3d) / 1e6, 'weights_mb': weight_size_mb(dense_path)},
]

size_df = pd.DataFrame(size_rows).sort_values(['weights_mb', 'params_m']).reset_index(drop=True)
display(size_df)

,model,component,params_m,weights_mb
0,2.5D full,slice only,0.048802,0.193631
1,2.5D split,fusion only,0.060675,0.235179
2,2.5D split,slice only,0.305762,1.174186
3,3D dense,full network,0.313507,1.204949
4,2.5D split,slice + fusion,0.366437,1.409365


In [7]:
def register_25d_full(moving, fixed):
    comps = [run_slice_orientation(slice_full, moving, fixed, ori) for ori in ORIENTATIONS]
    flows = [c[0] for c in comps]
    counts = [c[1] for c in comps]
    fused_flow = fuse_flow_components(flows, counts)
    moved = intensity_warper(moving, fused_flow)
    return moved, fused_flow

def register_25d_split(moving, fixed):
    comps = [run_slice_orientation(slice_split, moving, fixed, ori) for ori in ORIENTATIONS]
    flows = [c[0] for c in comps]
    counts = [c[1] for c in comps]
    fused_flow = fuse_flow_components(flows, counts)
    fusion_input = torch.cat([flows[0][:, :2], flows[1][:, :2], flows[2][:, :2]], dim=1)
    fusion_flow = fusion_split(fusion_input)
    total_flow = fused_flow + fusion_flow
    moved = intensity_warper(moving, total_flow)
    return moved, total_flow

def register_3d_dense(moving, fixed):
    return dense_3d(moving, fixed)

In [8]:
def evaluate_pipeline(name, register_fn):
    detail_rows = []
    per_label = {}
    pairs = [(0, 1), (1, 0)]
    for moving_idx, fixed_idx in pairs:
        moving = volumes_tensor[moving_idx:moving_idx+1]
        fixed = volumes_tensor[fixed_idx:fixed_idx+1]
        moving_seg = segmentations_tensor[moving_idx:moving_idx+1]
        fixed_seg = segmentations_tensor[fixed_idx:fixed_idx+1]
        with torch.no_grad():
            moved, flow = register_fn(moving, fixed)
        mse = F.mse_loss(moved, fixed).item()
        ncc_score = ncc(moved, fixed)
        smooth = flow_grad_l2_3d(flow).item()
        dice_map = dice_from_flow(flow, moving_seg, fixed_seg, label_ids, label_warper)
        valid_dice = [v for k, v in dice_map.items() if k != 0]
        metrics = {
            'model': name,
            'pair': f'{moving_idx}->{fixed_idx}',
            'mse': mse,
            'ncc': ncc_score,
            'smoothness': smooth,
            'dice_mean': float(np.mean(valid_dice)) if valid_dice else float('nan'),
            'dice_min': float(np.min(valid_dice)) if valid_dice else float('nan'),
            'dice_max': float(np.max(valid_dice)) if valid_dice else float('nan'),
            'dice_background': dice_map.get(0, float('nan')),
        }
        detail_rows.append(metrics)
        per_label[f'{moving_idx}->{fixed_idx}'] = dice_map
    detail_df = pd.DataFrame(detail_rows)
    summary = detail_df.groupby('model').mean(numeric_only=True).reset_index()
    summary['pair'] = 'mean'
    return detail_df, summary, per_label

In [9]:
pipelines = {
    '2.5D full (slice fusion)': register_25d_full,
    '2.5D split (slice + fusion)': register_25d_split,
    '3D dense Vxm': register_3d_dense,
}

detailed_frames = []
summary_frames = []
dice_breakdowns = {}

for name, fn in pipelines.items():
    detail_df, summary_df, per_label = evaluate_pipeline(name, fn)
    detailed_frames.append(detail_df)
    summary_frames.append(summary_df)
    dice_breakdowns[name] = per_label

detailed_results = pd.concat(detailed_frames, ignore_index=True)
summary_results = pd.concat(summary_frames, ignore_index=True)
summary_sorted = summary_results.sort_values('dice_mean', ascending=False).reset_index(drop=True)

display(detailed_results.sort_values(['model', 'pair']).reset_index(drop=True))
display(summary_sorted)

print('Stored per-label Dice maps in `dice_breakdowns`.')

,model,pair,mse,ncc,smoothness,dice_mean,dice_min,dice_max,dice_background
0,2.5D full (slice fusion),0->1,0.001324,0.945978,0.022484,0.553612,0.000000,0.903027,0.974490
1,2.5D full (slice fusion),1->0,0.001274,0.948570,0.022954,0.565922,0.000000,0.896113,0.974220
2,2.5D split (slice + fusion),0->1,0.000466,0.970331,0.082605,0.688104,0.166771,0.931122,0.982287
3,2.5D split (slice + fusion),1->0,0.000455,0.971719,0.082464,0.677986,0.084125,0.917744,0.981892
4,3D dense Vxm,0->1,0.000679,0.964358,0.033863,0.678208,0.156250,0.931641,0.979866
5,3D dense Vxm,1->0,0.000669,0.965800,0.033577,0.699286,0.141145,0.927646,0.979406


,model,mse,ncc,smoothness,dice_mean,dice_min,dice_max,dice_background,pair
0,3D dense Vxm,0.000674,0.965079,0.033720,0.688747,0.148698,0.929643,0.979636,mean
1,2.5D split (slice + fusion),0.000461,0.971025,0.082534,0.683045,0.125448,0.924433,0.982090,mean
2,2.5D full (slice fusion),0.001299,0.947274,0.022719,0.559767,0.000000,0.899570,0.974355,mean


Stored per-label Dice maps in `dice_breakdowns`.


### Key comparison (sorted by Dice mean)

In [10]:
size_lookup = {
    '2.5D full (slice fusion)': size_df.query("model == '2.5D full' and component == 'slice only'").iloc[0],
    '2.5D split (slice + fusion)': size_df.query("model == '2.5D split' and component == 'slice + fusion'").iloc[0],
    '3D dense Vxm': size_df.query("model == '3D dense'").iloc[0],
}

comparison_rows = []
for _, row in summary_sorted.iterrows():
    size_row = size_lookup[row['model']]
    comparison_rows.append({
        'model': row['model'],
        'dice_mean': row['dice_mean'],
        'dice_min': row['dice_min'],
        'dice_max': row['dice_max'],
        'mse': row['mse'],
        'ncc': row['ncc'],
        'smoothness': row['smoothness'],
        'params_m': size_row['params_m'],
        'weights_mb': size_row['weights_mb'],
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('dice_mean', ascending=False).reset_index(drop=True)
display(comparison_df)

,model,dice_mean,dice_min,dice_max,mse,ncc,smoothness,params_m,weights_mb
0,3D dense Vxm,0.688747,0.148698,0.929643,0.000674,0.965079,0.033720,0.313507,1.204949
1,2.5D split (slice + fusion),0.683045,0.125448,0.924433,0.000461,0.971025,0.082534,0.366437,1.409365
2,2.5D full (slice fusion),0.559767,0.000000,0.899570,0.001299,0.947274,0.022719,0.048802,0.193631


Use the helper below (or access `dice_breakdowns` directly) to inspect class-wise Dice scores for any model/pair.

In [11]:
example_model = '2.5D split (slice + fusion)'
print('Per-label Dice for', example_model)
pd.DataFrame(dice_breakdowns[example_model]).T

Per-label Dice for 2.5D split (slice + fusion)


,0.0,2.0,3.0,4.0,5.0,7.0,8.0,10.0,11.0,12.0,...,60.0,62.0,63.0,77.0,85.0,251.0,252.0,253.0,254.0,255.0
0->1,0.982287,0.823839,0.710419,0.853675,0.537205,0.830555,0.883628,0.870425,0.829922,0.865486,...,0.823251,0.294118,0.166771,0.219323,0.586538,0.631431,0.382542,0.408969,0.640359,0.643580
1->0,0.981892,0.829765,0.706360,0.802290,0.528302,0.835181,0.884391,0.882441,0.838964,0.860918,...,0.818812,0.153846,0.084125,0.304230,0.524851,0.552751,0.382589,0.510268,0.640867,0.685046
